# 🛍️ FeedbackIQ — Google Colab Setup
**Customer Feedback Intelligence System**  
Aspect-Based Sentiment Analysis + RAG Chatbot powered by local HuggingFace models (no API keys needed)

---
### Steps:
1. Install dependencies
2. Clone / mount the project
3. Download & preprocess data
4. Launch Streamlit via `localtunnel`

> **Tip:** Use `Runtime > Change runtime type > T4 GPU` for faster inference.

## 1️⃣ Install Dependencies

In [ ]:
# Install all project dependencies
!pip install -q datasets pandas pyarrow numpy scikit-learn joblib \
             transformers sentence-transformers faiss-cpu \
             streamlit matplotlib seaborn nbformat nbconvert pytest \
             google-genai openai

# Install tunnel to expose Streamlit
!npm install -g -q localtunnel

print('✅ All dependencies installed!')

## 2️⃣ Mount Google Drive & Clone Project

In [ ]:
import os

USE_DRIVE = True  # Set False to clone from GitHub instead

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # ⚠️ Update this path to where your FeedbakIQ folder is on your Drive
    PROJECT_PATH = '/content/drive/MyDrive/FeedbakIQ'
    if not os.path.exists(PROJECT_PATH):
        print('❌ Folder not found on Drive. Please upload FeedbakIQ to Google Drive')
        print('   OR set USE_DRIVE = False to clone from GitHub.')
    else:
        print(f'✅ Project found at {PROJECT_PATH}')
else:
    # Clone from GitHub (update URL to your repo)
    GITHUB_URL = 'https://github.com/YOUR_USERNAME/FeedbakIQ.git'  # ← Update this
    !git clone {GITHUB_URL} /content/FeedbakIQ
    PROJECT_PATH = '/content/FeedbakIQ'
    print(f'✅ Cloned to {PROJECT_PATH}')

os.chdir(PROJECT_PATH)
import sys
sys.path.insert(0, PROJECT_PATH)
print(f'Working directory: {os.getcwd()}')

## 3️⃣ Download & Preprocess Data (skip if already done)

In [ ]:
import os

reviews_path = os.path.join(PROJECT_PATH, 'data/processed/combined_processed.parquet')

if os.path.exists(reviews_path):
    print('✅ Processed data already exists — skipping download.')
else:
    print('📥 Downloading & processing data (takes ~5-10 min)...')
    !python src/data_loader.py
    !python src/preprocessing.py
    print('✅ Data pipeline complete!')

## 4️⃣ Verify Data Loaded Correctly

In [ ]:
import pandas as pd

df = pd.read_parquet('data/processed/combined_processed.parquet')
print(f'✅ Reviews loaded: {len(df):,}')
print(f'Columns: {list(df.columns)}')
print(f"\nSentiment distribution:")
print(df['sentiment'].value_counts())
print(f"\nDomains:")
print(df['domain'].value_counts())

## 5️⃣ Configure LLM Backend

Since Ollama doesn't run in Colab, we use the built-in **HuggingFace local model** (Flan-T5) or the **Grounded Synthesizer** (no model needed).

In [ ]:
# Patch the Streamlit app to use HuggingFace local LLM instead of Ollama
# Options: 'grounded' (fastest, no model) | 'hf' (flan-t5-small, GPU recommended)

LLM_BACKEND = 'hf'  # Change to 'grounded' for instant load, no model needed
HF_MODEL = 'google/flan-t5-small'  # or 'google/flan-t5-base' for better quality

# Write config patch
patch = f"""
import os, sys
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
sys.path.insert(0, '{PROJECT_PATH}')
"""

print(f'✅ LLM backend configured: {LLM_BACKEND} ({HF_MODEL if LLM_BACKEND == "hf" else "no model"})')

## 6️⃣ Launch Streamlit App

In [ ]:
import subprocess, threading, time, requests

# Patch the app to use HF backend instead of Ollama
app_patch = '''
import os, sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
sys.path.insert(0, "{PROJECT_PATH}")
'''.format(PROJECT_PATH=PROJECT_PATH)

# Read and patch the streamlit app to use selected backend
with open('app/streamlit_app.py', 'r') as f:
    app_code = f.read()

# Replace Ollama default with selected backend in Colab context
if LLM_BACKEND == 'grounded':
    app_code_patched = app_code.replace(
        '_llm = OllamaLLM(model_name=ollama_model)',
        '_llm = GroundedSynthesizerLLM()'
    ).replace(
        'index=0,  # Ollama default',
        'index=1,  # Grounded default in Colab'
    )
else:
    app_code_patched = app_code

with open('/tmp/feedbackiq_app.py', 'w') as f:
    f.write(app_code_patched)

print('Starting Streamlit...')
proc = subprocess.Popen(
    ['streamlit', 'run', 'app/streamlit_app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

time.sleep(5)
print('✅ Streamlit started (PID:', proc.pid, ')')

In [ ]:
# Get public URL via localtunnel
import subprocess, time

lt_proc = subprocess.Popen(
    ['lt', '--port', '8501'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)
time.sleep(3)
line = lt_proc.stdout.readline()
print('🌐 Public URL:', line.strip())
print()
print('👆 Open the URL above in a new tab to access FeedbackIQ!')
print('   (If prompted for a password, enter your Colab IP shown at https://ipv4.icanhazip.com)')

## 7️⃣ (Optional) Get Colab IP for localtunnel password
If localtunnel asks for a password, run this cell to get your IP.

In [ ]:
!curl -s https://ipv4.icanhazip.com

---
## 📌 Notes
| Setting | Value |
|---|---|
| Streamlit port | 8501 |
| LLM Backend | HuggingFace `flan-t5-small` (no API key) |
| Embeddings | `all-MiniLM-L6-v2` (auto-downloaded) |
| Data | Amazon Reviews 2023 (92k reviews) |

**To use Grounded Synthesizer (instant, no model):** Set `LLM_BACKEND = 'grounded'` in cell 5.

**To use a better model:** Set `HF_MODEL = 'google/flan-t5-base'` (requires GPU).